# Silver shared reference data

Runs once per pipeline run, before the per-source Silver transforms. Maintains the conformed, source-independent tables
that every source transform uses:

| Table | Purpose |
|---|---|
| `DataSource` | One row per source system: category, integration method, data domains, cross-source dedupe priority, enabled flag and notebooks (from Bronze `meta.source_registry`) |
| `ActivityTypeMapping` | Source activity types (Strava, Apple Health, Garmin) -> canonical activity type and family |
| `HealthMetric` | Catalogue of health metrics used by `HealthObservation` |
| `SleepSession`, `SleepStage`, `NutritionIntake`, `UserGoal`, `UserPreference` | Contract tables, created empty once |

In [ ]:
rebuild_tables = False

In [ ]:
from datetime import datetime, timezone
from pyspark.sql import functions as F
import json

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.parquet.vorder.default", "true")

WORKSPACE_ID = notebookutils.runtime.context["currentWorkspaceId"]
def lakehouse_id(name):
    """Resolve a lakehouse in this workspace by name, so the notebook works in any workspace."""
    try:
        item = notebookutils.lakehouse.get(name)
        return item["id"] if isinstance(item, dict) else item.id
    except Exception:
        import sempy.fabric as fabric
        return fabric.resolve_item_id(name, type="Lakehouse")


BRONZE_TABLES = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{lakehouse_id('Bronze')}/Tables"
PROCESSED_AT = datetime.now(timezone.utc).replace(microsecond=0)
row_counts = {}


def table_exists(name):
    return notebookutils.fs.exists(f"Tables/{name}/_delta_log")


def read_table(name):
    return spark.read.format("delta").load(f"Tables/{name}")


def write_table(df, name):
    """Full overwrite of a shared reference table (written by path to keep the PascalCase name)."""
    (df.withColumn("silver_processed_at", F.lit(PROCESSED_AT).cast("timestamp"))
       .write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"Tables/{name}"))
    row_counts[name] = read_table(name).count()
    print(f"{name}: {row_counts[name]} rows")


def create_contract_table(name, ddl):
    if rebuild_tables or not table_exists(name):
        spark.createDataFrame([], ddl).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"Tables/{name}")
    row_counts[name] = read_table(name).count()
    print(f"{name}: contract table ready ({row_counts[name]} rows)")

In [ ]:
# Reference data: data sources, activity type mapping (Strava + sample Apple Health / Garmin mappings), metric catalogue

DATA_SOURCE_CATALOGUE = spark.createDataFrame([
    ("strava", "Strava", "fitness_platform", "oauth_rest_api", '["workouts","workout_timeseries","training_zones","equipment","body_measurements"]', 30, True),
    ("garmin_connect", "Garmin Connect", "wearable_platform", "garmin_health_api", '["workouts","workout_timeseries","sleep","daily_activity","biometrics","body_measurements"]', 10, False),
    ("apple_health", "Apple Health", "health_aggregator", "healthkit_ios", '["workouts","workout_timeseries","sleep","daily_activity","biometrics","body_measurements","nutrition"]', 20, False),
    ("oura", "Oura", "wearable_platform", "oauth_rest_api", '["sleep","biometrics","daily_activity"]', 15, False),
    ("whoop", "WHOOP", "wearable_platform", "oauth_rest_api", '["sleep","biometrics","workouts"]', 15, False),
    ("manual", "Manual entry", "manual", "app_input", '["workouts","body_measurements","nutrition"]', 50, False),
    ("app", "iOS app", "application", "app_backend", '["goals","nutrition","subjective_wellbeing"]', 40, False),
], "source_system string, source_name string, source_category string, integration_method string, data_domains_json string, workout_priority int, is_active boolean")

# DataSource = static catalogue enriched with the Bronze control table meta.source_registry (enabled flag, Bronze schema, notebooks)
registry_path = f"{BRONZE_TABLES}/meta/source_registry"
if notebookutils.fs.exists(f"{registry_path}/_delta_log"):
    registry = spark.read.format("delta").load(registry_path).select(
        "source_system", F.col("source_name").alias("registry_source_name"), "bronze_schema", "ingest_notebook",
        "silver_notebook", F.col("is_enabled").alias("registry_enabled"), "load_order")
else:
    registry = spark.createDataFrame([], "source_system string, registry_source_name string, bronze_schema string, "
                                         "ingest_notebook string, silver_notebook string, registry_enabled boolean, load_order int")
ref_data_source = (
    DATA_SOURCE_CATALOGUE.join(registry, "source_system", "full")
    .select(
        "source_system",
        F.coalesce("source_name", "registry_source_name", "source_system").alias("source_name"),
        F.coalesce("source_category", F.lit("other")).alias("source_category"),
        F.coalesce("integration_method", F.lit("unknown")).alias("integration_method"),
        "data_domains_json",
        F.coalesce("workout_priority", F.lit(90)).alias("workout_priority"),
        F.coalesce("registry_enabled", "is_active", F.lit(False)).alias("is_active"),
        "bronze_schema", "ingest_notebook", "silver_notebook",
        F.coalesce("load_order", F.lit(999)).alias("load_order"),
    )
)

# (source_activity_type, activity_type, activity_family, is_endurance, is_distance_based, is_indoor_default, primary_intensity_metric)
STRAVA_TYPES = [
    ("Ride", "cycling_road", "cycling", True, True, False, "power"),
    ("VirtualRide", "cycling_virtual", "cycling", True, True, True, "power"),
    ("GravelRide", "cycling_gravel", "cycling", True, True, False, "power"),
    ("MountainBikeRide", "cycling_mountain", "cycling", True, True, False, "power"),
    ("EBikeRide", "cycling_ebike", "cycling", True, True, False, "heart_rate"),
    ("EMountainBikeRide", "cycling_ebike", "cycling", True, True, False, "heart_rate"),
    ("Handcycle", "cycling_handcycle", "cycling", True, True, False, "power"),
    ("Velomobile", "cycling_velomobile", "cycling", True, True, False, "power"),
    ("Run", "running_road", "running", True, True, False, "pace"),
    ("TrailRun", "running_trail", "running", True, True, False, "heart_rate"),
    ("VirtualRun", "running_treadmill", "running", True, True, True, "pace"),
    ("Walk", "walking", "walking", True, True, False, "heart_rate"),
    ("Hike", "hiking", "walking", True, True, False, "heart_rate"),
    ("Swim", "swimming", "swimming", True, True, False, "pace"),
    ("Rowing", "rowing", "rowing", True, True, False, "power"),
    ("VirtualRow", "rowing_indoor", "rowing", True, True, True, "power"),
    ("Canoeing", "paddling", "paddling", True, True, False, "heart_rate"),
    ("Kayaking", "paddling", "paddling", True, True, False, "heart_rate"),
    ("StandUpPaddling", "paddling", "paddling", True, True, False, "heart_rate"),
    ("WeightTraining", "strength_training", "strength", False, False, True, "heart_rate"),
    ("Crossfit", "functional_training", "strength", False, False, True, "heart_rate"),
    ("HighIntensityIntervalTraining", "hiit", "conditioning", False, False, True, "heart_rate"),
    ("Workout", "general_workout", "conditioning", False, False, True, "heart_rate"),
    ("Elliptical", "elliptical", "cardio_machine", True, False, True, "heart_rate"),
    ("StairStepper", "stair_stepper", "cardio_machine", True, False, True, "heart_rate"),
    ("Yoga", "yoga", "mobility", False, False, True, "heart_rate"),
    ("Pilates", "pilates", "mobility", False, False, True, "heart_rate"),
    ("RockClimbing", "climbing", "climbing", False, False, False, "heart_rate"),
    ("AlpineSki", "skiing_alpine", "winter_sport", False, True, False, "heart_rate"),
    ("BackcountrySki", "skiing_backcountry", "winter_sport", True, True, False, "heart_rate"),
    ("NordicSki", "skiing_nordic", "winter_sport", True, True, False, "heart_rate"),
    ("RollerSki", "skiing_roller", "winter_sport", True, True, False, "heart_rate"),
    ("Snowboard", "snowboarding", "winter_sport", False, True, False, "heart_rate"),
    ("Snowshoe", "snowshoeing", "winter_sport", True, True, False, "heart_rate"),
    ("IceSkate", "skating_ice", "skating", True, True, False, "heart_rate"),
    ("InlineSkate", "skating_inline", "skating", True, True, False, "heart_rate"),
    ("Skateboard", "skateboarding", "skating", False, True, False, "heart_rate"),
    ("Golf", "golf", "sport", False, True, False, "heart_rate"),
    ("Soccer", "soccer", "team_sport", False, True, False, "heart_rate"),
    ("Tennis", "tennis", "racket_sport", False, False, False, "heart_rate"),
    ("Badminton", "badminton", "racket_sport", False, False, True, "heart_rate"),
    ("Squash", "squash", "racket_sport", False, False, True, "heart_rate"),
    ("Racquetball", "racquetball", "racket_sport", False, False, True, "heart_rate"),
    ("Pickleball", "pickleball", "racket_sport", False, False, False, "heart_rate"),
    ("TableTennis", "table_tennis", "racket_sport", False, False, True, "heart_rate"),
    ("Surfing", "surfing", "water_sport", False, False, False, "heart_rate"),
    ("Windsurf", "windsurfing", "water_sport", False, True, False, "heart_rate"),
    ("Kitesurf", "kitesurfing", "water_sport", False, True, False, "heart_rate"),
    ("Sail", "sailing", "water_sport", False, True, False, "heart_rate"),
    ("Wheelchair", "wheelchair", "other", True, True, False, "heart_rate"),
]
APPLE_HEALTH_TYPES = [
    ("HKWorkoutActivityTypeCycling", "cycling_road", "cycling", True, True, False, "power"),
    ("HKWorkoutActivityTypeRunning", "running_road", "running", True, True, False, "pace"),
    ("HKWorkoutActivityTypeWalking", "walking", "walking", True, True, False, "heart_rate"),
    ("HKWorkoutActivityTypeHiking", "hiking", "walking", True, True, False, "heart_rate"),
    ("HKWorkoutActivityTypeSwimming", "swimming", "swimming", True, True, False, "pace"),
    ("HKWorkoutActivityTypeRowing", "rowing", "rowing", True, True, False, "power"),
    ("HKWorkoutActivityTypeTraditionalStrengthTraining", "strength_training", "strength", False, False, True, "heart_rate"),
    ("HKWorkoutActivityTypeFunctionalStrengthTraining", "functional_training", "strength", False, False, True, "heart_rate"),
    ("HKWorkoutActivityTypeHighIntensityIntervalTraining", "hiit", "conditioning", False, False, True, "heart_rate"),
    ("HKWorkoutActivityTypeYoga", "yoga", "mobility", False, False, True, "heart_rate"),
    ("HKWorkoutActivityTypePilates", "pilates", "mobility", False, False, True, "heart_rate"),
    ("HKWorkoutActivityTypeElliptical", "elliptical", "cardio_machine", True, False, True, "heart_rate"),
]
GARMIN_TYPES = [
    ("cycling", "cycling_road", "cycling", True, True, False, "power"),
    ("road_biking", "cycling_road", "cycling", True, True, False, "power"),
    ("indoor_cycling", "cycling_virtual", "cycling", True, True, True, "power"),
    ("virtual_ride", "cycling_virtual", "cycling", True, True, True, "power"),
    ("gravel_cycling", "cycling_gravel", "cycling", True, True, False, "power"),
    ("mountain_biking", "cycling_mountain", "cycling", True, True, False, "power"),
    ("running", "running_road", "running", True, True, False, "pace"),
    ("treadmill_running", "running_treadmill", "running", True, True, True, "pace"),
    ("trail_running", "running_trail", "running", True, True, False, "heart_rate"),
    ("walking", "walking", "walking", True, True, False, "heart_rate"),
    ("hiking", "hiking", "walking", True, True, False, "heart_rate"),
    ("lap_swimming", "swimming", "swimming", True, True, True, "pace"),
    ("open_water_swimming", "swimming_open_water", "swimming", True, True, False, "pace"),
    ("strength_training", "strength_training", "strength", False, False, True, "heart_rate"),
    ("hiit", "hiit", "conditioning", False, False, True, "heart_rate"),
    ("yoga", "yoga", "mobility", False, False, True, "heart_rate"),
    ("indoor_rowing", "rowing_indoor", "rowing", True, True, True, "power"),
]
ACTIVITY_TYPE_DDL = ("source_activity_type string, activity_type string, activity_family string, is_endurance boolean, "
                     "is_distance_based boolean, is_indoor_default boolean, primary_intensity_metric string")
ref_activity_type = (
    spark.createDataFrame(STRAVA_TYPES, ACTIVITY_TYPE_DDL).withColumn("source_system", F.lit("strava"))
    .unionByName(spark.createDataFrame(APPLE_HEALTH_TYPES, ACTIVITY_TYPE_DDL).withColumn("source_system", F.lit("apple_health")))
    .unionByName(spark.createDataFrame(GARMIN_TYPES, ACTIVITY_TYPE_DDL).withColumn("source_system", F.lit("garmin_connect")))
    .select("source_system", "source_activity_type", "activity_type", "activity_family", "is_endurance",
            "is_distance_based", "is_indoor_default", "primary_intensity_metric")
)

# (metric_code, metric_name, domain, canonical_unit, aggregation, higher_is_better, description)
METRICS = [
    ("body_weight_kg", "Body weight", "body_composition", "kg", "last", None, "Body mass"),
    ("body_fat_pct", "Body fat", "body_composition", "%", "last", False, "Body fat percentage"),
    ("lean_body_mass_kg", "Lean body mass", "body_composition", "kg", "last", True, "Lean body mass"),
    ("bmi", "Body mass index", "body_composition", "kg/m2", "last", None, "Body mass index"),
    ("height_cm", "Height", "body_composition", "cm", "last", None, "Body height"),
    ("waist_circumference_cm", "Waist circumference", "body_composition", "cm", "last", False, "Waist circumference"),
    ("resting_heart_rate_bpm", "Resting heart rate", "cardiovascular", "bpm", "avg", False, "Resting heart rate"),
    ("hrv_rmssd_ms", "HRV (RMSSD)", "recovery", "ms", "avg", True, "Heart rate variability, RMSSD"),
    ("hrv_sdnn_ms", "HRV (SDNN)", "recovery", "ms", "avg", True, "Heart rate variability, SDNN (Apple Health)"),
    ("vo2max_ml_kg_min", "VO2max", "cardiovascular", "ml/kg/min", "last", True, "Estimated maximal oxygen uptake"),
    ("max_heart_rate_bpm", "Max heart rate", "cardiovascular", "bpm", "max", None, "Maximum heart rate"),
    ("lactate_threshold_hr_bpm", "Lactate threshold HR", "performance", "bpm", "last", None, "Lactate threshold heart rate"),
    ("ftp_watts", "Functional threshold power", "performance", "W", "last", True, "Functional threshold power"),
    ("respiratory_rate_brpm", "Respiratory rate", "cardiovascular", "breaths/min", "avg", None, "Respiratory rate"),
    ("spo2_pct", "Blood oxygen saturation", "cardiovascular", "%", "avg", True, "Blood oxygen saturation"),
    ("body_temperature_deviation_c", "Body temperature deviation", "recovery", "C", "avg", None, "Deviation from baseline skin/body temperature"),
    ("blood_pressure_systolic_mmhg", "Systolic blood pressure", "cardiovascular", "mmHg", "avg", False, "Systolic blood pressure"),
    ("blood_pressure_diastolic_mmhg", "Diastolic blood pressure", "cardiovascular", "mmHg", "avg", False, "Diastolic blood pressure"),
    ("blood_glucose_mmol_l", "Blood glucose", "metabolic", "mmol/L", "avg", None, "Blood glucose"),
    ("steps_count", "Steps", "daily_activity", "count", "sum", True, "Step count"),
    ("active_energy_kcal", "Active energy", "daily_activity", "kcal", "sum", True, "Active energy burned"),
    ("basal_energy_kcal", "Basal energy", "daily_activity", "kcal", "sum", None, "Resting / basal energy burned"),
    ("exercise_minutes", "Exercise minutes", "daily_activity", "min", "sum", True, "Exercise minutes"),
    ("stand_hours", "Stand hours", "daily_activity", "h", "sum", True, "Hours with standing activity"),
    ("walking_running_distance_m", "Walking + running distance", "daily_activity", "m", "sum", True, "Daily walking and running distance"),
    ("floors_climbed", "Floors climbed", "daily_activity", "count", "sum", True, "Floors climbed"),
    ("stress_score", "Stress score", "recovery", "score", "avg", False, "Vendor stress score (0-100)"),
    ("body_battery", "Body battery", "recovery", "score", "last", True, "Garmin body battery (0-100)"),
    ("readiness_score", "Readiness score", "recovery", "score", "last", True, "Vendor readiness/recovery score (0-100)"),
    ("mood_score", "Mood", "subjective_wellbeing", "score", "avg", True, "Self-reported mood (1-5)"),
    ("energy_level_score", "Energy level", "subjective_wellbeing", "score", "avg", True, "Self-reported energy (1-5)"),
    ("muscle_soreness_score", "Muscle soreness", "subjective_wellbeing", "score", "avg", False, "Self-reported soreness (1-5)"),
]
ref_metric = spark.createDataFrame(
    METRICS,
    "metric_code string, metric_name string, metric_domain string, canonical_unit string, daily_aggregation string, higher_is_better boolean, description string",
)

write_table(ref_data_source, "DataSource")
write_table(ref_activity_type, "ActivityTypeMapping")
write_table(ref_metric, "HealthMetric")

In [ ]:
# Contract tables for upcoming sources (Apple Health, Garmin, Oura, WHOOP) and the iOS app. Created empty once.

create_contract_table("SleepSession", (
    "sleep_session_id string, user_id string, source_system string, source_record_id string, "
    "sleep_start_utc timestamp, sleep_end_utc timestamp, sleep_start_local timestamp, local_date date, timezone_name string, "
    "time_in_bed_s int, asleep_s int, awake_s int, light_s int, deep_s int, rem_s int, sleep_latency_s int, "
    "awakenings_count int, sleep_efficiency_pct double, avg_heart_rate_bpm double, lowest_heart_rate_bpm double, "
    "avg_hrv_rmssd_ms double, avg_respiratory_rate_brpm double, avg_spo2_pct double, skin_temperature_deviation_c double, "
    "source_sleep_score double, is_nap boolean, device_name string, source_attributes_json string, "
    "ingested_at timestamp, silver_processed_at timestamp"))

create_contract_table("SleepStage", (
    "sleep_stage_id string, sleep_session_id string, user_id string, source_system string, "
    "stage string, stage_start_utc timestamp, stage_end_utc timestamp, duration_s int, ingested_at timestamp, silver_processed_at timestamp"))

create_contract_table("NutritionIntake", (
    "intake_id string, user_id string, source_system string, source_record_id string, consumed_at_utc timestamp, "
    "consumed_at_local timestamp, local_date date, meal_type string, food_name string, serving_description string, "
    "energy_kcal double, carbohydrate_g double, sugar_g double, fiber_g double, protein_g double, fat_g double, "
    "saturated_fat_g double, sodium_mg double, caffeine_mg double, alcohol_g double, water_ml double, "
    "entry_method string, source_attributes_json string, ingested_at timestamp, silver_processed_at timestamp"))

create_contract_table("UserGoal", (
    "goal_id string, user_id string, source_system string, goal_type string, goal_name string, "
    "target_metric_code string, baseline_value double, target_value double, target_unit string, "
    "start_date date, target_date date, priority int, status string, preferences_json string, "
    "created_at timestamp, updated_at timestamp, silver_processed_at timestamp"))

create_contract_table("UserPreference", (
    "user_id string, source_system string, preference_key string, preference_value string, "
    "updated_at timestamp, silver_processed_at timestamp"))

In [ ]:
result = {"status": "success", "layer": "silver_reference", "processed_at": PROCESSED_AT.isoformat(), "row_counts": row_counts}
print(json.dumps(result, indent=2))
notebookutils.notebook.exit(json.dumps(result))